In [ ]:
# -*- coding: utf-8 -*-
# ======================================================================
# ЭКСПЕРИМЕНТ 1: ДАННЫЕ БЕЗ ШУМА
# ======================================================================
# Цель: проверить, как нейросеть работает на идеально точных данных.
# Ожидается максимальная точность, особенно для параметра a2.
# ======================================================================

# 1. ИМПОРТ НЕОБХОДИМЫХ БИБЛИОТЕК
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Фиксируем случайные числа, чтобы результаты были воспроизводимы
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# 2. ФУНКЦИЯ, ВЫЧИСЛЯЮЩАЯ ТЕОРЕТИЧЕСКУЮ КРИВУЮ ПОЛЗУЧЕСТИ
def creep_strain(t, a0, a1, a2):
    return a0 + a1 * np.exp(-a2 * t)

In [ ]:
# 3. ГЕНЕРАЦИЯ ДАННЫХ БЕЗ ШУМА
def generate_dataset_no_noise(n_samples=200, t_points=None):
    if t_points is None:
        t_points = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 1.5, 2.0])
    X_list, y_list = [], []
    for _ in range(n_samples):
        a0 = np.random.uniform(0.6, 1.5)
        a1 = np.random.uniform(-0.8, -0.2)
        a2 = np.random.uniform(2.0, 8.0)
        # Проверка физичности
        if a0 + a1 <= 0.05:
            continue
        eps_vals = creep_strain(t_points, a0, a1, a2)
        if np.any(eps_vals <= 0):
            continue
        # Шум НЕ добавляем (noise_level = 0)
        X_list.append(eps_vals)
        y_list.append([a0, a1, a2])
    return np.vstack(X_list), np.vstack(y_list), t_points

print(">>> Эксперимент 1: Генерация данных БЕЗ ШУМА")
t_grid = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 1.0, 1.5, 2.0])
X, y, t_grid = generate_dataset_no_noise(n_samples=200)
print(f"    Создано {X.shape[0]} образцов (шум = 0%).")

# Сохраняем в CSV
df = pd.DataFrame(X, columns=[f"eps_t{t}" for t in t_grid])
df[['a0','a1','a2']] = y
df.to_csv("experiment1_no_noise.csv", index=False)
print(">>> Данные сохранены в 'experiment1_no_noise.csv'.")

# Визуализация трёх кривых на густой сетке
t_fine = np.linspace(0, 2, 200)
plt.figure(figsize=(9, 5))
for i in range(3):
    a0, a1, a2 = y[i]
    eps_smooth = creep_strain(t_fine, a0, a1, a2)
    plt.plot(t_fine, eps_smooth, '-', linewidth=1.5, label=f'Образец {i+1}')
    plt.plot(t_grid, X[i], 'o', markersize=6)
plt.xlabel('Время t, с')
plt.ylabel('Деформация ε')
plt.title('Эксперимент 1: Кривые ползучести БЕЗ ШУМА')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ НЕЙРОСЕТИ
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

batch_size = 16
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 5. АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (такая же, как в основном коде)
class CreepParameterNet(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[32, 16, 8]):
        super(CreepParameterNet, self).__init__()
        layers = []
        prev_size = input_dim
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            layers.append(nn.ReLU())
            prev_size = h_size
        layers.append(nn.Linear(prev_size, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = CreepParameterNet(X_train.shape[1])
print("\n>>> Архитектура нейросети:")
print(model)

In [ ]:
# 6. НАСТРОЙКА ОБУЧЕНИЯ
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 800
train_loss_history = []
test_loss_history = []

print("\n>>> Начало обучения (Эксперимент 1: без шума)...")
for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * batch_X.size(0)

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    train_loss_history.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)
        test_loss = criterion(test_pred, y_test_t).item()
        test_loss_history.append(test_loss)

    if (epoch + 1) % 100 == 0:
        print(f"Эпоха {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.6f} | Test Loss: {test_loss:.6f}")

# График обучения
plt.figure(figsize=(8, 5))
plt.plot(train_loss_history, label='Ошибка на обучении')
plt.plot(test_loss_history, label='Ошибка на тесте')
plt.xlabel('Эпоха')
plt.ylabel('MSE Loss')
plt.title('Эксперимент 1: Динамика обучения (без шума)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 7. ТЕСТИРОВАНИЕ И ОЦЕНКА КАЧЕСТВА
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).numpy()
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = y_test

mse = np.mean((y_true - y_pred) ** 2)
r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
mae = mean_absolute_error(y_true, y_pred)

print("\n>>> РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА 1 (без шума):")
print(f"    Среднеквадратичная ошибка (MSE) : {mse:.6f}")
print(f"    Коэффициент детерминации R²   : {r2:.4f}")
print(f"    Средняя абсолютная ошибка (MAE): {mae:.6f}")

param_names = ['a0', 'a1', 'a2']
for i, name in enumerate(param_names):
    r2_i = r2_score(y_true[:, i], y_pred[:, i])
    mae_i = mean_absolute_error(y_true[:, i], y_pred[:, i])
    print(f"    {name}: R² = {r2_i:.4f}, MAE = {mae_i:.4f}")

# Графики "Истина vs Предсказание"
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(param_names):
    ax = axes[i]
    ax.scatter(y_true[:, i], y_pred[:, i], alpha=0.7)
    min_val = min(y_true[:, i].min(), y_pred[:, i].min())
    max_val = max(y_true[:, i].max(), y_pred[:, i].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')
    ax.set_xlabel(f'Истинное значение {name}')
    ax.set_ylabel(f'Предсказанное значение {name}')
    ax.set_title(f'Параметр {name}')
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 8. ВОССТАНОВЛЕНИЕ КРИВОЙ
print("\n>>> Эксперимент 1: Восстановление кривой для тестового образца (без шума)")

sample_idx = 0   # можно изменить, чтобы посмотреть другие образцы

# Истинные параметры – из тестовой выборки
true_params = y_true[sample_idx]
# Предсказанные параметры – что выдала сеть для этого образца
pred_params = y_pred[sample_idx]
# "Измерения" (без шума) – вход сети для этого образца
eps_points = X_test[sample_idx]   # в тестовой выборке шума тоже нет

print(f"    Истинные параметры:     a0 = {true_params[0]:.4f}, a1 = {true_params[1]:.4f}, a2 = {true_params[2]:.4f}")
print(f"    Предсказанные параметры: a0 = {pred_params[0]:.4f}, a1 = {pred_params[1]:.4f}, a2 = {pred_params[2]:.4f}")

# Строим гладкие кривые
t_fine = np.linspace(0.1, 2.0, 200)
eps_true_curve = creep_strain(t_fine, *true_params)
eps_pred_curve = creep_strain(t_fine, *pred_params)

plt.figure(figsize=(9, 5))
plt.plot(t_fine, eps_true_curve, 'b-', linewidth=2, label='Истинная кривая')
plt.plot(t_fine, eps_pred_curve, 'r--', linewidth=2, label='Предсказанная кривая')
plt.plot(t_grid, eps_points, 'ko', markersize=8, label='Точки измерений (без шума)')
plt.xlabel('Время t, с')
plt.ylabel('Деформация ε')
plt.title('Эксперимент без шума: проверка на тестовом образце')
plt.legend()
plt.grid(True)
plt.show()